In [ ]:
import os

import contextily as ctx
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from matplotlib import colors

In [ ]:
def bitwiseMask(ds): # HERE
    '''
    This function masks a PIXC granule: for now, it ony remove pixels
    with land classification and those with bad geolocation_qual.
    # See page 65 of PIXC PDD: https://podaac.jpl.nasa.gov/SWOT?tab=datasets-information&sections=about%2Bdata
    '''
    mask = np.where((ds.classification > 1) &
                    (ds.interferogram_qual < 2**16) &
                    (ds.classification_qual < 2**16) &
                    (ds.geolocation_qual < 2**16) &
                    (ds.sig0_qual < 2**16) &
                    (np.abs(ds.cross_track) > 10000) &
                    (np.abs(ds.cross_track) < 60000))[0]
    
    print(mask.shape)
    return mask

In [ ]:
def makeGDF(ds, mask, data_vars):
    '''
    This function takes the pixel cloud xarray object, makes a masked
    GeoDataFrame, renames columns as needed, set the CRS, reprojects
    the CRS, and returns a GeoDataFrame.
    '''

    # Subset xarray, convert to masked DataFrame
    xarr = ds[data_vars]
    df = xarr.to_dataframe().loc[mask].reset_index()

    # Create GDF
    gdf_PIXC = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df.longitude, df.latitude),
        crs="EPSG:4326"
    ) # PIXC has no native CRS, setting same as River_SP

    if 'classification' in gdf_PIXC.columns:
        gdf_PIXC.rename(columns={'classification': 'klass'}, inplace=True)
    
    # Convert the crs to WGS 84 / UTM zone 18N
    gdf_PIXC = gdf_PIXC.to_crs(epsg='3857')

    return gdf_PIXC

### NHD Waterbody

In [ ]:
index = 4

data_path = '/nas/cee-ice/data/NHDPlus_HR/'

## Set-up
mdata_path = '/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC/data/'
# Max binsize of 1000 is plenty for CONUS w/o lakes
# Mississippi is ~472 m wide at mouth as calculated here
bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 150, 200, 500, 1000]

In [ ]:
# Define dtypes for lookup tables to preserve leading zeros
dtype_dic= {'HUC4': str, 'HUC2': str, 'toBasin': str, 'level': str}
# Read in HUC lookup table
lookup = pd.read_csv(os.path.join(mdata_path,
                                  'HUC4_lookup_no_great_lakes.csv'),
                     dtype=dtype_dic)

# Get current HUC2 and HUC4 IDs
huc2 = f'HUC2_{lookup.loc[index,'HUC4'][0:2]}'
huc4 = f'NHDPLUS_H_{lookup.loc[index,'HUC4']}_HU4_GDB'

# Set data filepath
file_path = os.path.join(data_path, huc2, huc4, huc4 + '.gdb')

In [ ]:
area = gpd.read_file(filename=file_path, layer='NHDArea',
                      engine='pyogrio')

area = area.to_crs(epsg='32618')

### Pixel Cloud

#### Original PIXC granule

In [ ]:
data_dir = '/nas/cee-ice/data/SWOT/PIXC_vD/'
# pixc_path = os.path.join(data_dir, 'SWOT_L2_HR_PIXC_031_298_080L_20250417T185908_20250417T185919_PIC2_01.nc')
pixc_path = os.path.join(data_dir, 'SWOT_L2_HR_PIXC_032_298_080L_20250508T154411_20250508T154422_PID0_02.nc')
tile_name = pixc_path[-71:-3]

# Read in xarray
ds_PIXC = xr.open_dataset(filename_or_obj=pixc_path, group='pixel_cloud', engine='h5netcdf')

In [ ]:
# Make mask
mask = bitwiseMask(ds_PIXC)

if mask.shape[0] == 0:
    print('This granule has no pixels after masking, exiting.')
    # sys.exit(1)    

In [ ]:
# Set desired data vars

variables = [
    'range_index', 
             'azimuth_index',
    # 'data_window_first_valid',
    # 'data_window_first_cross_track',
             'classification']

In [ ]:
# Convert PIXC to GeoDataFrame
gdf_PIXC = makeGDF(ds=ds_PIXC, mask=mask, data_vars=variables)

# Convert CRS to WGS 84 / UTM zone 18N
gdf_PIXC = gdf_PIXC.to_crs(epsg='32618')

In [ ]:
# # Connecticut
# bounds = [694000, 4688500, 699000, 4695166] # 5 km x 6.67 km

# PIXC_c = gdf_PIXC.clip(mask=bounds)

In [ ]:
# Make dict for legend labels
flags = ds_PIXC.classification.flag_meanings.split() # extract each flag meaning
codes = {idx:k for idx, k in enumerate(flags, start=1)}

In [ ]:
codes

#### Prepped pseudo-pixels

In [ ]:
data_dir = '/nas/cee-water/cjgleason/fiona/for_presentations/2025_AGU/data/'

# pseudo_path = os.path.join(data_dir, 'SWOT_L2_HR_PIXC_032_298_080L_20250508T154411_20250508T154422_PID0_02_pseudo.parquet')
pseudo_path = os.path.join(data_dir, 'SWOT_L2_HR_PIXC_041_298_080L_20251112T102954_20251112T103005_PID0_02_pseudo.parquet')
# pseudo_path = os.path.join(data_dir, 'SWOT_L2_HR_PIXC_012_298_080L_20240317T084237_20240317T084248_PIC0_01_pseudo.parquet')

gdf_pseudo = gpd.read_parquet(pseudo_path)

# Convert CRS to WGS 84 / UTM zone 18N
gdf_pseudo = gdf_pseudo.to_crs(epsg='32618')

# tile_name = pseudo_path[-71:-3]

### NHDPlus HR

In [ ]:
huc = '0108'

data_dir = '/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_data/NHD_prepped_buffered/HUC2_01/'
file_path = f'{data_dir}NHDPLUS_H_{huc}_HU4_GDB_prepped_buffered_mean.parquet'

buffer = gpd.read_parquet(file_path)

# Convert CRS to WGS 84 / UTM zone 18N
buffer = buffer.to_crs(epsg='32618')

In [ ]:
huc = '0108'

data_dir = '/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_data/NHD_prepped_buffered_extra/HUC2_01/'
file_path = f'{data_dir}NHDPLUS_H_{huc}_HU4_GDB_prepped_buffered_extra_mean.parquet'

extra = gpd.read_parquet(file_path)

# Convert CRS to WGS 84 / UTM zone 18N
extra = extra.to_crs(epsg='32618')

In [ ]:
save_path = '/nas/cee-water/cjgleason/fiona/for_presentations/2025_AGU/intro_figures/'

### NHDPlus HR Plots

In [ ]:
palette = {2: 'darkolivegreen', 3: 'cornflowerblue', 4: 'blue', 5: 'hotpink',
           6: 'darkkhaki', 7: 'springgreen'} # 6: 'lightsalmon', 7: 'lightblue'

In [ ]:
from shapely.geometry import box

In [ ]:
fort = buffer[buffer['NHDPlusID'] == 10000900053916]

In [ ]:
fort_extra = extra[extra['NHDPlusID'] == 10000900053916]

In [ ]:
gdf_pseudo_cor = gpd.sjoin(
    gdf_pseudo, fort_extra, how='inner',
    predicate='within').reset_index().drop(columns=['index', 'index_right']
)

In [ ]:
# Invisible bounding polygon
bbox_poly = gpd.GeoDataFrame(
    geometry=[box(bounds[0], bounds[1], bounds[2], bounds[3])], 
    crs=fort.crs
)

In [ ]:
bounds = [700700, 4690820, 701190, 4691100] # 5 km x 6.67 km

gdf_PIXC_c = gdf_PIXC.clip(mask=fort_extra)
nums = list(np.unique(gdf_PIXC_c.klass).astype('int'))
labels = [codes[x] for x in nums]

# Make cmap
cmap = colors.ListedColormap([palette[x] for x in nums])

fig, ax = plt.subplots(figsize=(5,3))
# basin.clip(mask=bound_box).plot(ax=ax, alpha=0.5)
fort.plot(ax=ax, alpha=0.7, color='k')
gdf_PIXC_c.plot(ax=ax, column=gdf_PIXC_c.klass,
              categorical=True, cmap=cmap, s=9)

# Plot invisible bounding box BEFORE add_basemap
bbox_poly.plot(ax=ax, alpha=0)  # invisible but sets extent
# Basemap
ctx.add_basemap(ax, crs=gdf_PIXC_c.crs,
                source=ctx.providers.CartoDB.Positron,
                zoom=17, alpha=0.9)

plt.xlim(700700, 701190)
plt.ylim(4690820, 4691100);
plt.axis('off');

# plt.savefig(f'{save_path}fort_PIXC.png', dpi=300, bbox_inches='tight') 
plt.savefig(os.path.join(save_path, 'fort_PIXC.svg'), format='svg', bbox_inches='tight')

In [ ]:
bounds = [700700, 4690820, 701190, 4691100] # 5 km x 6.67 km

nums = list(np.unique(gdf_pseudo_cor.klass).astype('int'))
labels = [codes[x] for x in nums]

# Make cmap
cmap = colors.ListedColormap([palette[x] for x in nums])

fig, ax = plt.subplots(figsize=(5,3))
# basin.clip(mask=bound_box).plot(ax=ax, alpha=0.5)
fort.plot(ax=ax, alpha=0.7, color='k')
gdf_pseudo_cor.plot(ax=ax, alpha=0.9, column=gdf_pseudo_cor.klass,
              categorical=True, cmap=cmap)

# Plot invisible bounding box BEFORE add_basemap
bbox_poly.plot(ax=ax, alpha=0)  # invisible but sets extent
# Basemap
ctx.add_basemap(ax, crs=gdf_pseudo_cor.crs, source=ctx.providers.CartoDB.Positron,
                zoom=17, alpha=0.9)

plt.xlim(700700, 701190)
plt.ylim(4690820, 4691100);
plt.axis('off');

# plt.savefig(f'{save_path}fort_pseudo.png', dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(save_path, 'fort_pseudo.svg'), format='svg', bbox_inches='tight')

In [ ]:
gdf_pseudo_dissolve = gdf_pseudo_cor.dissolve()

In [ ]:
bounds = [700700, 4690820, 701190, 4691100] # 5 km x 6.67 km

fig, ax = plt.subplots(figsize=(5,3))
# basin.clip(mask=bound_box).plot(ax=ax, alpha=0.5)
fort.plot(ax=ax, alpha=0.7, color='k')
gdf_pseudo_dissolve.plot(ax=ax, color='c')

# Plot invisible bounding box BEFORE add_basemap
bbox_poly.plot(ax=ax, alpha=0)  # invisible but sets extent
# Basemap
ctx.add_basemap(ax, crs=gdf_pseudo_cor.crs,
                source=ctx.providers.CartoDB.Positron,
                zoom=17, alpha=0.9)

plt.xlim(700700, 701190)
plt.ylim(4690820, 4691100);
plt.axis('off');

# plt.savefig(f'{save_path}fort_dissolve.png', dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(save_path, 'fort_dissolve.svg'), format='svg', bbox_inches='tight')

In [ ]:
gdf_pseudo_clip = gdf_pseudo_dissolve.clip(fort)

In [ ]:
bounds = [700700, 4690820, 701190, 4691100] # 5 km x 6.67 km

fig, ax = plt.subplots(figsize=(5,3))
# basin.clip(mask=bound_box).plot(ax=ax, alpha=0.5)
fort.plot(ax=ax, alpha=0.7, color='k')
gdf_pseudo_clip.plot(ax=ax, color='c')
# Plot invisible bounding box BEFORE add_basemap
bbox_poly.plot(ax=ax, alpha=0)  # invisible but sets extent
# Basemap
ctx.add_basemap(ax, crs=gdf_pseudo_clip.crs,
                source=ctx.providers.CartoDB.Positron,
                zoom=17, alpha=0.9)

plt.xlim(700700, 701190)
plt.ylim(4690820, 4691100);
plt.axis('off');

# plt.savefig(f'{save_path}fort_clip.png', dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(save_path, 'fort_clip.svg'), format='svg', bbox_inches='tight')